# Cargar y preparar datos

In [9]:
# Para que funcionen los imports
import sys
import os
# 
# Añadir la raíz del proyecto al path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.append(ROOT)

print('Project root:', ROOT)

Project root: c:\Users\urkow\Documents\Documentos_Clase\Caraduras_ML


In [6]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE = (360, 360)
BATCH_SIZE = 32
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)


In [ ]:
# Cargar datasets
train_ds = tf.keras.utils.image_dataset_from_directory(
    ROOT+'/data/train',
    label_mode='binary',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    ROOT+'/data/test',
    label_mode='binary',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

Found 1029 files belonging to 2 classes.
Found 33 files belonging to 2 classes.


Data augmentation

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

Class Weights (compensar desbalance)

In [17]:
# Contar imágenes por clase en train
cara_count = len(os.listdir(ROOT+'/data/train/cara'))
sincara_count = len(os.listdir(ROOT+'/data/train/sin-cara'))

labels = np.array([0]*sincara_count + [1]*cara_count)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0,1]),
    y=labels
)

class_weights = {0: class_weights[0], 1: class_weights[1]}
class_weights

{0: 0.7935285053929122, 1: 1.3517060367454068}

# Construcción y entrenamiento de modelo

In [18]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,)
)
base_model.trainable = False  # fase 1: congelado

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")]
)

model.summary()


16705208/16705208 [==============================] - 1s 0us/step
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 360, 360, 3)]     0         
                                                                 
 sequential (Sequential)     (None, 360, 360, 3)       0         
                                                                 
 efficientnetb0 (Functional  (None, 12, 12, 1280)      4049571   
 )                                                               
                                                                 
 global_average_pooling2d (  (None, 1280)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                             

Entrenamos modelo

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_auc"),
    keras.callbacks.ModelCheckpoint("best_model.h5", save_best_only=True, monitor="val_auc")
]

history = model.fit(
    train_ds,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks,
    validation_split = 0.2
)
